In [2]:
# -*- coding: utf-8 -*-
"""
Physics-Informed Machine Learning (PIML) for Slope Stability
============================================================
Pipeline: Rain → θ (PIML + reduced Richards) → ψ (Van Genuchten) → u → FoS

Figures produced (publication-quality, 300 dpi):
  Fig 1  — Loss Convergence (Train vs Val, log-scale)
  Fig 2  — Scatter: Train / Val / Test R² (3-panel)
  Fig 3  — θ Time-series: Observed vs Predicted (full dataset)
  Fig 4  — Scatter Residuals (Test set)
  Fig 5  — Van Genuchten Characteristic Curves (θ–ψ and θ–Kr)
  Fig 6  — Matric Suction Time-series
  Fig 7  — Pore Pressure & FoS Time-series (combined)
  Fig 8  — FoS Distribution Histogram
  Fig 9  — Physics Residual dθ/dt Time-series (PINN proof)
  Fig 10 — Full Dashboard (5-panel overview)
  Model  — Saved as piml_slope_model.pt (state_dict + metadata)
"""

import warnings; warnings.filterwarnings("ignore")
import os, json, datetime
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ── Reproducibility ─────────────────────────────────────────
torch.manual_seed(42); np.random.seed(42)
DEVICE = torch.device("cpu")
print(f"Device: {DEVICE}")

# ── Output directory ────────────────────────────────────────
OUT = "/content/piml_figures"
os.makedirs(OUT, exist_ok=True)

# ── Global plot style (publication-ready) ───────────────────
STYLE = {
    "font.family"        : "DejaVu Sans",
    "font.size"          : 10,
    "axes.labelsize"     : 11,
    "axes.titlesize"     : 12,
    "axes.titleweight"   : "bold",
    "axes.spines.top"    : False,
    "axes.spines.right"  : False,
    "axes.linewidth"     : 0.8,
    "xtick.major.size"   : 3.5,
    "ytick.major.size"   : 3.5,
    "legend.frameon"     : False,
    "legend.fontsize"    : 9,
    "figure.dpi"         : 300,
    "savefig.dpi"        : 300,
    "savefig.bbox"       : "tight",
    "savefig.facecolor"  : "white",
}
plt.rcParams.update(STYLE)

C = {
    "rain"   : "#4895EF",
    "obs"    : "#E63946",
    "pred"   : "#2EC4B6",
    "train"  : "#3A86FF",
    "val"    : "#FF006E",
    "test"   : "#FB5607",
    "psi"    : "#7209B7",
    "u"      : "#F77F00",
    "fos"    : "#06D6A0",
    "warn"   : "#EF233C",
    "grid"   : "#CCCCCC",
    "resid"  : "#480CA8",
    "phys"   : "#023E8A",
    "vg1"    : "#0077B6",
    "vg2"    : "#00B4D8",
}

def savefig(fig, name, tight=True):
    path = os.path.join(OUT, name)
    fig.savefig(path, dpi=300, bbox_inches="tight" if tight else None,
                facecolor="white")
    plt.close(fig)
    print(f"  → Saved: {path}")
    return path

def add_panel_label(ax, label, x=-0.08, y=1.06):
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=13, fontweight="bold", va="top", ha="right")

def light_grid(ax, axis="y"):
    ax.grid(axis=axis, color=C["grid"], lw=0.5, ls="--", alpha=0.7)

# ═══════════════════════════════════════════════════════════
# 1-2. LOAD & PREPROCESS  (Dev108 only)
# ═══════════════════════════════════════════════════════════
print("\n[Step 1-2] Loading data ...")
df_raw = pd.read_csv("/content/pinn_108_new.csv", low_memory=False)
df_raw = df_raw.dropna(subset=["timestamp","devID","soil","rain"])
df_raw["devID"] = df_raw["devID"].astype(int)
df_raw["timestamp"] = pd.to_datetime(df_raw["timestamp"], dayfirst=False, errors="coerce")
df_raw = df_raw.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

# ── Dev108 only ──────────────────────────────────────────────
df = df_raw[df_raw["devID"]==108].copy().reset_index(drop=True)
df = df.rename(columns={"soil":"theta"})
df = df[["timestamp","theta","rain"]].sort_values("timestamp").reset_index(drop=True)

df["t_sec"] = (df["timestamp"] - df["timestamp"].iloc[0]).dt.total_seconds()
df = df.dropna(subset=["rain","theta","t_sec"]).reset_index(drop=True)

# ── [FIX 1] Sensor artifact removal ─────────────────────────
# Data analysis: 200 rows (0.32%) have theta < 0.10
# These are sensor startup artifacts causing VG blowup (ψ → 1e12)
# Observed clean range: 0.39–0.54 m³/m³
n_before = len(df)
df = df[df["theta"] > 0.30].reset_index(drop=True)
print(f"  Artifact rows removed: {n_before - len(df)} (theta ≤ 0.30)")

df = df.iloc[::2].reset_index(drop=True)
print(f"  Rows after subsample : {len(df):,}")

# ── Chronological split on raw (un-smoothed) data ───────────
LAG   = 1
n_raw = len(df)
n_tr_raw  = int(0.80 * n_raw)
n_val_raw = int(0.90 * n_raw)

df_tr  = df.iloc[:n_tr_raw].copy().reset_index(drop=True)
df_val = df.iloc[n_tr_raw:n_val_raw].copy().reset_index(drop=True)
df_te  = df.iloc[n_val_raw:].copy().reset_index(drop=True)

# ── Apply rolling median per split (no cross-boundary leakage) ──
ROLL_W = 5
for _df in [df_tr, df_val, df_te]:
    _df["theta"] = _df["theta"].rolling(ROLL_W, center=False, min_periods=1).median()

# ── Build features (lagged theta + rain) per split ──────────
def build_features(d, lag=1):
    X_rows, y_rows = [], []
    for i in range(lag, len(d)):
        lags = [d["theta"].iloc[i-k] for k in range(1, lag+1)]
        X_rows.append([d["rain"].iloc[i]] + lags)
        y_rows.append(d["theta"].iloc[i])
    X = np.array(X_rows, dtype=np.float32)
    y = np.array(y_rows, dtype=np.float32).reshape(-1, 1)
    rain_  = d["rain"].values[lag:].astype(np.float32)
    theta_ = d["theta"].values[lag:].astype(np.float32)
    t_sec_ = d["t_sec"].values[lag:].astype(np.float32)
    ts_    = d["timestamp"].values[lag:]
    return X, y, rain_, theta_, t_sec_, ts_

X_tr_raw,  y_tr_raw,  rain_tr,  theta_tr,  t_sec_tr,  ts_tr  = build_features(df_tr,  LAG)
X_val_raw, y_val_raw, rain_val, theta_val, t_sec_val, ts_val  = build_features(df_val, LAG)
X_te_raw,  y_te_raw,  rain_te,  theta_te,  t_sec_te,  ts_te   = build_features(df_te,  LAG)

# ── Scalers fit on TRAIN only ────────────────────────────────
scaler_X = MinMaxScaler().fit(X_tr_raw)
scaler_y = MinMaxScaler().fit(y_tr_raw)

X_tr  = scaler_X.transform(X_tr_raw).astype(np.float32)
X_val = scaler_X.transform(X_val_raw).astype(np.float32)
X_te  = scaler_X.transform(X_te_raw).astype(np.float32)
y_tr  = scaler_y.transform(y_tr_raw).astype(np.float32)
y_val = scaler_y.transform(y_val_raw).astype(np.float32)
y_te  = scaler_y.transform(y_te_raw).astype(np.float32)

# Combined arrays for full-dataset plotting
X_np     = np.concatenate([X_tr_raw, X_val_raw, X_te_raw],  axis=0)
theta_np = np.concatenate([theta_tr, theta_val, theta_te],   axis=0)
rain_np  = np.concatenate([rain_tr,  rain_val,  rain_te],    axis=0)
t_sec_np = np.concatenate([t_sec_tr, t_sec_val, t_sec_te],   axis=0)
ts_all_np= np.concatenate([ts_tr,    ts_val,    ts_te],      axis=0)
X_norm   = scaler_X.transform(X_np).astype(np.float32)

def to_t(a): return torch.tensor(a).to(DEVICE)
Xt,yt,Xv,yv,Xte,yte = to_t(X_tr),to_t(y_tr),to_t(X_val),to_t(y_val),to_t(X_te),to_t(y_te)
rain_t = to_t(rain_tr[:,None])

dt_tr = np.diff(t_sec_tr, prepend=t_sec_tr[0]).clip(min=1.0)
dt_t  = to_t(dt_tr[:,None])

# ═══════════════════════════════════════════════════════════
# 3. MODEL
# ═══════════════════════════════════════════════════════════
print("\n[Step 3] Building PIML model ...")

class PIML_MLP(nn.Module):
    def __init__(self, in_dim, h=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim,h), nn.Tanh(),
            nn.Linear(h,h),      nn.Tanh(),
            nn.Linear(h,h//2),   nn.Tanh(),
            nn.Linear(h//2,1),   nn.Sigmoid())
        self.log_alpha_D = nn.Parameter(torch.tensor(-1.0))
    def forward(self,x): return self.net(x)
    def drainage(self,theta_n):
        return torch.exp(self.log_alpha_D).clamp(0.0001,5.0) * theta_n

model = PIML_MLP(in_dim=X_tr.shape[1]).to(DEVICE)
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

# ═══════════════════════════════════════════════════════════
# 4-6. TRAIN
# ═══════════════════════════════════════════════════════════
print("\n[Step 4-6] Training ...")
EPOCHS=2000; LR=5e-4; LAM=3; BATCH=2048

H = 1.5

RAIN_MAX    = float(rain_tr.max()) + 1e-8   # train only
THETA_SCALE = float(theta_tr.max() - theta_tr.min()) + 1e-8   # train only

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
mse = nn.MSELoss()

dataset = TensorDataset(Xt, yt, rain_t, dt_t)
loader  = DataLoader(dataset, batch_size=BATCH, shuffle=False)
train_losses, val_losses = [], []
best_val=np.inf; best_state=None

for epoch in range(1, EPOCHS+1):
    model.train()
    ep_loss = 0.0
    prev_last = None
    for Xb, yb, rb, dtb in loader:
        optimizer.zero_grad()
        pred = model(Xb)
        loss_d = mse(pred, yb)

        # ── [FIX] Correct batch-boundary temporal continuity ──
        if prev_last is None:
            pred_prev = torch.cat([pred[:1].detach(), pred[:-1]], dim=0)
        else:
            pred_prev = torch.cat([prev_last, pred[:-1]], dim=0)
        prev_last = pred[-1:].detach()

        dt_scaled  = dtb  + 1e-8
        theta_scale_t = torch.tensor(THETA_SCALE, dtype=torch.float32)
        theta_min_t   = torch.tensor(float(scaler_y.data_min_[0]), dtype=torch.float32)
        pred_real      = pred * theta_scale_t + theta_min_t
        pred_prev_real = pred_prev * theta_scale_t + theta_min_t
        dthdt_pred = (pred_real - pred_prev_real) / dt_scaled


        I_n = (rb / 1000.0) / 3600.0 / H
        D_n = model.drainage(pred_real) / 3600.0


        res   = dthdt_pred - (I_n - D_n)
        loss_p = (res**2).mean()

        loss = loss_d + LAM * loss_p
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        ep_loss += loss.item()

    scheduler.step()
    model.eval()
    with torch.no_grad():
        vl = mse(model(Xv), yv).item()
    train_losses.append(ep_loss/len(loader))
    val_losses.append(vl)
    if vl < best_val:
        best_val=vl; best_state={k:v.clone() for k,v in model.state_dict().items()}
    if epoch%100==0:
        print(f"  Epoch {epoch:4d} | Train {train_losses[-1]:.5f} | Val {vl:.5f} | "
              f"αD={torch.exp(model.log_alpha_D).item():.4f}")

model.load_state_dict(best_state)

# ═══════════════════════════════════════════════════════════
# MODEL SAVE
# ═══════════════════════════════════════════════════════════
print("\n[Model Save] Saving model ...")
model_path = os.path.join(OUT, "piml_slope_model.pt")
torch.save({
    "model_state_dict"  : best_state,
    "model_architecture": {"class":"PIML_MLP","in_dim":X_tr.shape[1],"hidden":64},
    "hyperparameters"   : {"epochs":EPOCHS,"lr":LR,"lambda_phys":LAM,
                           "batch_size":BATCH,"lag":LAG},
    "scalers": {
        "scaler_X_min"  : scaler_X.data_min_.tolist(),
        "scaler_X_scale": scaler_X.scale_.tolist(),
        "scaler_y_min"  : scaler_y.data_min_.tolist(),
        "scaler_y_scale": scaler_y.scale_.tolist(),
    },
    # ── [FIX 2] Corrected VG parameters ──────────────────────
    # Source: Carsel & Parrish (1988), Sandy Clay Loam
    # theta_s=0.57: observed max=0.54 + 0.03 buffer (data-driven)
    "vg_params"  : dict(theta_r=0.065, theta_s=0.57, alpha=1.9, n=1.31, m=1-1/1.31),
    "slope_params": dict(slope_deg=25.0, c_kPa=5.0, phi_deg=28.0, gamma_s=18.5, H_m=1.5),
    "train_losses"   : train_losses,
    "val_losses"     : val_losses,
    "best_val_loss"  : best_val,
    "alpha_D_learned": torch.exp(model.log_alpha_D).item(),
    "timestamp"      : datetime.datetime.now().isoformat(),
}, model_path)
print(f"  → Model saved: {model_path}")

# ═══════════════════════════════════════════════════════════
# 7. EVALUATE
# ═══════════════════════════════════════════════════════════
print("\n[Step 7] Evaluate ...")
model.eval()
with torch.no_grad():
    pred_tr_norm  = model(Xt).cpu().numpy()
    pred_val_norm = model(Xv).cpu().numpy()
    pred_te_norm  = model(Xte).cpu().numpy()
    pred_all_norm = model(to_t(X_norm)).cpu().numpy()

theta_pred_tr  = scaler_y.inverse_transform(pred_tr_norm).flatten()
theta_pred_val = scaler_y.inverse_transform(pred_val_norm).flatten()
theta_pred_te  = scaler_y.inverse_transform(pred_te_norm).flatten()
theta_pred_all = scaler_y.inverse_transform(pred_all_norm).flatten()
theta_obs_all  = theta_np.copy()

r2_tr  = r2_score(theta_tr,  theta_pred_tr)
r2_val = r2_score(theta_val, theta_pred_val)
r2_te  = r2_score(theta_te,  theta_pred_te)
rmse_te= np.sqrt(mean_squared_error(theta_te, theta_pred_te))
residuals = theta_te - theta_pred_te

print(f"  Train R²: {r2_tr:.4f}  |  Val R²: {r2_val:.4f}  |  Test R²: {r2_te:.4f}")
print(f"  Test RMSE: {rmse_te:.5f} m³/m³")

rain_all  = rain_np.copy()
ts = pd.to_datetime(ts_all_np)

# ═══════════════════════════════════════════════════════════
# 8. Van Genuchten  θ → ψ
# ═══════════════════════════════════════════════════════════
print("\n[Step 8] Van Genuchten θ→ψ ...")

# ── [FIX 2] Corrected VG parameters ──────────────────────────
# Carsel & Parrish (1988) Sandy Clay Loam:
#   theta_r=0.065, alpha=0.019 cm⁻¹=1.9 m⁻¹, n=1.31
# theta_s=0.57: data-driven (observed max=0.54 + 0.03 buffer)
VG = dict(theta_r=0.065, theta_s=0.57, alpha=1.9, n=1.31, m=1-1/1.31)
print(f"  VG: θ_r={VG['theta_r']}  θ_s={VG['theta_s']}  "
      f"α={VG['alpha']} m⁻¹  n={VG['n']}")

def vg_psi(theta, vg):
    # ── [FIX 3] Three-layer protection against numerical blowup ──
    # Layer 1: theta clipping with adequate margin
    tc = np.clip(theta, vg["theta_r"] + 0.01, vg["theta_s"] - 0.01)
    # Layer 2: Se direct clipping
    Se = (tc - vg["theta_r"]) / (vg["theta_s"] - vg["theta_r"])
    Se = np.clip(Se, 0.01, 0.99)
    # Compute h
    h  = (1/vg["alpha"]) * (Se**(-1/vg["m"]) - 1)**(1/vg["n"])
    # Layer 3: physical cap — field matric suction rarely > 15m for clay loam
    h  = np.clip(h, 0.0, 15.0)
    return -h

def vg_Kr(theta, vg):
    tc = np.clip(theta, vg["theta_r"] + 0.01, vg["theta_s"] - 0.01)
    Se = (tc - vg["theta_r"]) / (vg["theta_s"] - vg["theta_r"])
    Se = np.clip(Se, 0.01, 0.99)
    return Se**0.5 * (1-(1-Se**(1/vg["m"]))**vg["m"])**2

psi_all = vg_psi(theta_pred_all, VG)

# ── Diagnostic print ─────────────────────────────────────────
print(f"  ψ range: {psi_all.min():.3f} to {psi_all.max():.3f} m")
print(f"  theta_pred range: {theta_pred_all.min():.4f} to {theta_pred_all.max():.4f}")

# VG curve arrays for Fig 5
theta_vg = np.linspace(VG["theta_r"]+0.005, VG["theta_s"]-0.005, 300)
psi_vg   = vg_psi(theta_vg, VG)
kr_vg    = vg_Kr(theta_vg, VG)

# ═══════════════════════════════════════════════════════════
# 9. Pore pressure & FoS
# ═══════════════════════════════════════════════════════════
gamma_w = 9.81
u_all   = gamma_w * psi_all   # kPa

SLOPE=25.0; beta=np.radians(SLOPE)
c_=5.0; phi_=np.radians(28.0)
gamma_s=18.5; H=1.5
sigma_n = gamma_s*H*np.cos(beta)**2
tau_d   = gamma_s*H*np.sin(beta)*np.cos(beta)
FoS_all = (c_ + np.maximum(sigma_n - u_all, 0)*np.tan(phi_)) / (tau_d+1e-8)

print(f"  u  range : {u_all.min():.2f} to {u_all.max():.2f} kPa")
print(f"  FoS min={FoS_all.min():.3f}  mean={FoS_all.mean():.3f}  "
      f"max={FoS_all.max():.3f}")
print(f"  FoS<1.0 events: {(FoS_all<1.0).sum()} | FoS<1.3 events: {(FoS_all<1.3).sum()}")

# ── Physics residual (per-split gradient — no leakage) ───────
n1 = len(theta_tr)
n2 = n1 + len(theta_val)
dthdt_pred_full = np.concatenate([
    np.gradient(theta_pred_all[:n1],  t_sec_np[:n1]),
    np.gradient(theta_pred_all[n1:n2],t_sec_np[n1:n2]),
    np.gradient(theta_pred_all[n2:],  t_sec_np[n2:]),
])
dthdt_obs = np.concatenate([
    np.gradient(theta_obs_all[:n1],  t_sec_np[:n1]),
    np.gradient(theta_obs_all[n1:n2],t_sec_np[n1:n2]),
    np.gradient(theta_obs_all[n2:],  t_sec_np[n2:]),
])

rain_norm_all = (rain_all / 1000.0) / 3600.0 / H    # train-only normalisation
alpha_D = torch.exp(model.log_alpha_D).item()
D_full  = alpha_D * (theta_pred_all - VG["theta_r"]) / 3600.0
phys_residual = dthdt_pred_full - (rain_norm_all - D_full)

# ═══════════════════════════════════════════════════════════
# FIGURES
# ═══════════════════════════════════════════════════════════
print("\n[Step 11] Generating publication-quality figures ...")

n_tr_plot  = len(theta_tr)
n_val_plot = n_tr_plot + len(theta_val)

# ──────────────────────────────────────────────────────────────
# FIG 1 — Loss Convergence
# ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7,4.5))
epochs_arr = np.arange(1, EPOCHS+1)
ax.semilogy(epochs_arr, train_losses, color=C["train"], lw=1.8, label="Training Loss")
ax.semilogy(epochs_arr, val_losses,   color=C["val"],   lw=1.8, label="Validation Loss", ls="--")
best_ep = int(np.argmin(val_losses))+1
ax.axvline(best_ep, color="gray", lw=1.0, ls=":", alpha=0.8,
           label=f"Best epoch ({best_ep})")
ax.annotate(f"Best val\n{min(val_losses):.5f}",
            xy=(best_ep, min(val_losses)),
            xytext=(best_ep+12, min(val_losses)*3),
            fontsize=8, arrowprops=dict(arrowstyle="->", color="gray", lw=0.8))
light_grid(ax)
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss (log scale)")
ax.set_title("PIML Training Loss Convergence")
ax.legend()
add_panel_label(ax, "(a)")
fig.text(0.5, -0.02,
         f"Cosine-annealed AdamW | λ_physics={LAM} | Batch={BATCH}",
         ha="center", fontsize=8, color="gray")
savefig(fig, "fig01_loss_convergence.png")

# ──────────────────────────────────────────────────────────────
# FIG 2 — Scatter: Train / Val / Test R²
# ──────────────────────────────────────────────────────────────
splits = [
    ("Train", theta_tr,  theta_pred_tr,  r2_tr,  C["train"]),
    ("Val",   theta_val, theta_pred_val, r2_val, C["val"]),
    ("Test",  theta_te,  theta_pred_te,  r2_te,  C["test"]),
]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, (label, obs, pred, r2, col) in zip(axes, splits):
    ax.scatter(obs, pred, s=6, alpha=0.35, color=col, rasterized=True)
    lim = [min(obs.min(), pred.min())-0.002, max(obs.max(), pred.max())+0.002]
    ax.plot(lim, lim, "k--", lw=1.2, label="1:1 line")
    m, b = np.polyfit(obs, pred, 1)
    xfit = np.linspace(*lim, 100)
    ax.plot(xfit, m*xfit+b, color=col, lw=1.4, ls="-", alpha=0.8, label="OLS fit")
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("θ Observed (m³/m³)")
    ax.set_ylabel("θ Predicted (m³/m³)")
    ax.set_title(f"{label} Set\nR² = {r2:.4f}")
    ax.legend(fontsize=8)
    light_grid(ax)
    rmse_s = np.sqrt(mean_squared_error(obs, pred))
    ax.text(0.05, 0.93, f"RMSE={rmse_s:.4f}", transform=ax.transAxes,
            fontsize=8, color="gray")
fig.suptitle("Soil Moisture Prediction: Train / Val / Test Scatter Plots",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
savefig(fig, "fig02_scatter_r2_splits.png")

# ──────────────────────────────────────────────────────────────
# FIG 3 — θ Time-series
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True,
                         gridspec_kw={"height_ratios":[1,3], "hspace":0.08})
axes[0].fill_between(ts, rain_all, color=C["rain"], alpha=0.7, step="mid")
axes[0].set_ylabel("Rainfall\n(mm/min)", labelpad=2)
axes[0].set_title("Soil Volumetric Water Content: Observed vs PIML Predicted", loc="left")
light_grid(axes[0])

axes[1].plot(ts, theta_obs_all,  color=C["obs"],  lw=1.2, alpha=0.9, label="θ Observed (Dev108)")
axes[1].plot(ts, theta_pred_all, color=C["pred"], lw=1.2, alpha=0.9, label="θ Predicted (PIML)", ls="--")
tr_end  = ts[n_tr_plot - 1]
val_end = ts[n_val_plot - 1]
axes[1].axvspan(ts[0], tr_end,   alpha=0.05, color=C["train"], label="Train")
axes[1].axvspan(tr_end, val_end, alpha=0.10, color=C["val"],   label="Val")
axes[1].axvspan(val_end, ts[-1], alpha=0.10, color=C["test"],  label="Test")
axes[1].axvline(tr_end,  color=C["train"], lw=0.8, ls=":", alpha=0.6)
axes[1].axvline(val_end, color=C["val"],   lw=0.8, ls=":", alpha=0.6)
axes[1].set_ylabel("θ (m³/m³)")
axes[1].set_xlabel("Timestamp")
axes[1].legend(ncol=2, fontsize=9)
light_grid(axes[1])
for ax in axes:
    ax.set_xlim(ts[0], ts[-1])
    for t in ax.get_xticklabels(): t.set_rotation(15)
fig.text(0.92, 0.28, f"Test R²={r2_te:.3f}\nRMSE={rmse_te:.4f}",
         fontsize=9, ha="right",
         bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", lw=0.8))
savefig(fig, "fig03_theta_timeseries.png")

# ──────────────────────────────────────────────────────────────
# FIG 4 — Residuals (Test set)
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ts_te_plot = ts[n_val_plot:]
axes[0].axhline(0, color="black", lw=0.8)
axes[0].fill_between(ts_te_plot, residuals, 0,
                     where=(residuals>=0), color=C["train"], alpha=0.5, label="Over-pred")
axes[0].fill_between(ts_te_plot, residuals, 0,
                     where=(residuals<0),  color=C["warn"],  alpha=0.5, label="Under-pred")
axes[0].set_xlabel("Timestamp"); axes[0].set_ylabel("Residual θ_obs − θ_pred (m³/m³)")
axes[0].set_title("Residuals vs Time (Test set)")
axes[0].legend(fontsize=8)
light_grid(axes[0])
for t in axes[0].get_xticklabels(): t.set_rotation(15)

axes[1].hist(residuals, bins=40, color=C["resid"], edgecolor="white", lw=0.3, alpha=0.85)
axes[1].axvline(0, color="black", lw=1.0, ls="--")
axes[1].axvline(residuals.mean(), color=C["warn"], lw=1.2, ls="--",
                label=f"Mean={residuals.mean():.4f}")
axes[1].axvline(residuals.std(),  color=C["val"],  lw=1.2, ls=":",
                label=f"σ={residuals.std():.4f}")
axes[1].axvline(-residuals.std(), color=C["val"],  lw=1.2, ls=":")
axes[1].set_xlabel("Residual (m³/m³)"); axes[1].set_ylabel("Count")
axes[1].set_title("Residual Distribution (Test set)")
axes[1].legend(fontsize=8)
light_grid(axes[1])
fig.suptitle("Scatter Residual Analysis: Test Set", fontsize=12, fontweight="bold")
plt.tight_layout()
savefig(fig, "fig04_scatter_residuals.png")

# ──────────────────────────────────────────────────────────────
# FIG 5 — Van Genuchten Characteristic Curves
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(-psi_vg, theta_vg, color=C["vg1"], lw=2.0, label="VG Model")
psi_te_abs = np.abs(vg_psi(theta_pred_te, VG))
sort_idx   = np.argsort(psi_te_abs)
axes[0].scatter(psi_te_abs[sort_idx[::5]], theta_pred_te[sort_idx[::5]],
                s=6, alpha=0.3, color=C["vg2"], rasterized=True, label="PIML θ̂ (test)")
axes[0].set_xlabel("|ψ|  Matric Suction (m)")
axes[0].set_ylabel("θ  Volumetric Water Content (m³/m³)")
axes[0].set_title("(a) Soil Water Characteristic Curve")
axes[0].set_xscale("log")
axes[0].legend()
light_grid(axes[0])
axes[0].text(0.98, 0.06,
             f"θ_r={VG['theta_r']}  θ_s={VG['theta_s']}\n"
             f"α={VG['alpha']} m⁻¹  n={VG['n']}\n"
             f"Source: Carsel & Parrish (1988) SCL",
             transform=axes[0].transAxes, fontsize=8, ha="right",
             bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", lw=0.7))

axes[1].plot(theta_vg, kr_vg, color=C["psi"], lw=2.0)
axes[1].set_xlabel("θ  Volumetric Water Content (m³/m³)")
axes[1].set_ylabel("K_r  Relative Hydraulic Conductivity (−)")
axes[1].set_title("(b) Relative Hydraulic Conductivity Curve")
light_grid(axes[1])
axes[1].fill_between(theta_vg, kr_vg, alpha=0.15, color=C["psi"])
fig.suptitle("Van Genuchten Soil Hydraulic Characteristic Curves",
             fontsize=12, fontweight="bold")
plt.tight_layout()
savefig(fig, "fig05_vg_characteristic_curves.png")

# ──────────────────────────────────────────────────────────────
# FIG 6 — Matric Suction Time-series
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True,
                         gridspec_kw={"height_ratios":[1,2,2], "hspace":0.1})
axes[0].fill_between(ts, rain_all, color=C["rain"], alpha=0.7, step="mid")
axes[0].set_ylabel("Rainfall\n(mm/min)")
axes[0].set_title("Matric Suction Derived via Van Genuchten from PIML θ̂", loc="left")
light_grid(axes[0])

axes[1].plot(ts, theta_pred_all, color=C["pred"], lw=1.1, alpha=0.9, label="θ Predicted")
axes[1].plot(ts, theta_obs_all,  color=C["obs"],  lw=0.8, alpha=0.5, ls=":", label="θ Observed")
axes[1].set_ylabel("θ (m³/m³)")
axes[1].set_title("Soil Moisture", loc="left", fontsize=10, fontweight="bold", pad=2)
axes[1].legend(fontsize=8)
light_grid(axes[1])

axes[2].plot(ts, -psi_all, color=C["psi"], lw=1.4)
axes[2].fill_between(ts, -psi_all, alpha=0.12, color=C["psi"])
axes[2].axhline(0, color="gray", lw=0.7, ls=":")
axes[2].set_ylabel("|ψ| (m)")
axes[2].set_xlabel("Timestamp")
axes[2].set_title("Matric Suction |ψ|", loc="left", fontsize=10, fontweight="bold", pad=2)
# ── robust y-limit ──
p2, p98 = np.percentile(-psi_all, [1, 99])
axes[2].set_ylim(max(0, p2-0.1), p98+0.5)
light_grid(axes[2])
for ax in axes:
    ax.set_xlim(ts[0], ts[-1])
    for t in ax.get_xticklabels(): t.set_rotation(15)
savefig(fig, "fig06_matric_suction_timeseries.png")

# ──────────────────────────────────────────────────────────────
# FIG 7 — Pore Pressure & FoS Time-series
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True,
                         gridspec_kw={"height_ratios":[1,2,2.5], "hspace":0.1})
axes[0].fill_between(ts, rain_all, color=C["rain"], alpha=0.7, step="mid")
axes[0].set_ylabel("Rainfall\n(mm/min)")
axes[0].set_title("Pore Water Pressure & Factor of Safety", loc="left")
light_grid(axes[0])

axes[1].plot(ts, u_all, color=C["u"], lw=1.3)
axes[1].axhline(0, color="gray", lw=0.7, ls=":")
axes[1].fill_between(ts, u_all, 0, where=(u_all>0),
                     color=C["warn"], alpha=0.3, label="Positive u (destabilising)")
axes[1].fill_between(ts, u_all, 0, where=(u_all<=0),
                     color=C["fos"], alpha=0.15, label="Negative u (stabilising)")
axes[1].set_ylabel("u  Pore Pressure (kPa)")
axes[1].legend(fontsize=8)
# ── robust y-limit ──
p2u, p98u = np.percentile(u_all, [1, 99])
axes[1].set_ylim(p2u - 2, p98u + 2)
light_grid(axes[1])

axes[2].plot(ts, FoS_all, color=C["fos"], lw=1.4, label="FoS (Infinite Slope)")
axes[2].axhline(1.0, color=C["warn"],    lw=1.5, ls="--", label="FoS = 1.0  ⚠ Failure threshold")
axes[2].axhline(1.3, color="darkorange", lw=1.0, ls=":",  label="FoS = 1.3  Warning level")
axes[2].fill_between(ts, FoS_all, 1.0, where=(FoS_all<1.0),
                     color=C["warn"], alpha=0.4,
                     label=f"Failure zone (n={(FoS_all<1.0).sum()})")
axes[2].set_ylabel("Factor of Safety (−)")
axes[2].set_xlabel("Timestamp")
axes[2].legend(ncol=2, fontsize=8)
# ── robust y-limit ──
p2f, p98f = np.percentile(FoS_all, [1, 99])
axes[2].set_ylim(max(0, p2f-0.2), min(10, p98f+0.5))
light_grid(axes[2])
for ax in axes:
    ax.set_xlim(ts[0], ts[-1])
    for t in ax.get_xticklabels(): t.set_rotation(15)
savefig(fig, "fig07_pore_pressure_fos_timeseries.png")

# ──────────────────────────────────────────────────────────────
# FIG 8 — FoS Distribution
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
FoS_plot = np.clip(FoS_all, 0, 10)   # display clip only
axes[0].hist(FoS_plot, bins=60, color=C["fos"], edgecolor="white", lw=0.3, alpha=0.85)
axes[0].axvline(1.0, color=C["warn"],   lw=2.0, ls="--",
                label=f"FoS=1.0  (n={(FoS_all<1.0).sum()})")
axes[0].axvline(1.3, color="darkorange",lw=1.5, ls=":",
                label=f"FoS=1.3  (n={(FoS_all<1.3).sum()})")
axes[0].axvline(FoS_all.mean(), color="black", lw=1.2, ls="-",
                label=f"Mean={FoS_all.mean():.3f}")
axes[0].set_xlabel("Factor of Safety (−)"); axes[0].set_ylabel("Count")
axes[0].set_title("(a) FoS Histogram")
axes[0].legend(fontsize=8)
light_grid(axes[0])

sorted_fos = np.sort(FoS_plot)
exceedance = 1 - np.arange(1, len(sorted_fos)+1) / len(sorted_fos)
axes[1].semilogy(sorted_fos, exceedance, color=C["fos"], lw=1.8)
axes[1].axvline(1.0, color=C["warn"],   lw=1.5, ls="--", label="FoS=1.0")
axes[1].axvline(1.3, color="darkorange",lw=1.0, ls=":",  label="FoS=1.3")
axes[1].set_xlabel("Factor of Safety (−)")
axes[1].set_ylabel("Exceedance Probability (log scale)")
axes[1].set_title("(b) FoS Exceedance Curve")
axes[1].legend(fontsize=8)
light_grid(axes[1])
fig.suptitle("Factor of Safety Distribution", fontsize=12, fontweight="bold")
plt.tight_layout()
savefig(fig, "fig08_fos_distribution.png")

# ──────────────────────────────────────────────────────────────
# FIG 9 — Physics Residual
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True,
                         gridspec_kw={"height_ratios":[1.5,1.5,2], "hspace":0.1})
axes[0].plot(ts, dthdt_pred_full, color=C["pred"], lw=1.0,
             label="dθ̂/dt — Model predicted rate")
axes[0].set_ylabel("dθ/dt (m³/m³/s)")
axes[0].set_title("PINN Physics Residual: dθ/dt = I(t) − D(θ̂)", loc="left")
axes[0].legend(fontsize=8)
light_grid(axes[0])

axes[1].plot(ts, rain_norm_all - D_full, color=C["rain"], lw=1.0,
             label="I(t) − D(θ̂)  (physics RHS)")
axes[1].set_ylabel("I − D (m³/m³/s)")
axes[1].legend(fontsize=8)
light_grid(axes[1])

axes[2].plot(ts, phys_residual, color=C["phys"], lw=0.8, alpha=0.8)
axes[2].axhline(0, color="gray", lw=0.7, ls=":")
axes[2].fill_between(ts, phys_residual, 0,
                     where=(np.abs(phys_residual)>2*np.std(phys_residual)),
                     color=C["warn"], alpha=0.5, label="High residual (>2σ)")
axes[2].set_ylabel("Physics Residual\ndθ̂/dt − (I−D)")
axes[2].set_xlabel("Timestamp")
axes[2].legend(fontsize=8)
light_grid(axes[2])
mu_r = phys_residual.mean(); sd_r = phys_residual.std()
axes[2].text(0.99, 0.97,
             f"μ = {mu_r:.2e}\nσ = {sd_r:.2e}\nαD = {alpha_D:.4f}",
             transform=axes[2].transAxes, fontsize=8, va="top", ha="right",
             bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", lw=0.7))
for ax in axes:
    ax.set_xlim(ts[0], ts[-1])
    for t in ax.get_xticklabels(): t.set_rotation(15)
savefig(fig, "fig09_pinn_physics_residual.png")

# ──────────────────────────────────────────────────────────────
# FIG 10 — Full Dashboard (5-panel)
# ──────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 17))
gs  = gridspec.GridSpec(5, 1, figure=fig, hspace=0.50)

ax0=fig.add_subplot(gs[0])
ax0.fill_between(ts, rain_all, color=C["rain"], alpha=0.75, step="mid")
ax0.set_ylabel("Rainfall (mm/min)")
ax0.set_title("A — Rainfall Intensity", fontweight="bold", loc="left")
light_grid(ax0); add_panel_label(ax0, "(A)", y=1.08)

ax1=fig.add_subplot(gs[1])
ax1.plot(ts, theta_obs_all,  color=C["obs"],  lw=1.2, alpha=0.85, label="θ Observed")
ax1.plot(ts, theta_pred_all, color=C["pred"], lw=1.2, alpha=0.85,
         label="θ Predicted (PIML)", ls="--")
ax1.set_ylabel("θ (m³/m³)")
ax1.set_title(f"B — Soil Moisture  [Test R²={r2_te:.3f}  RMSE={rmse_te:.4f} m³/m³]",
              fontweight="bold", loc="left")
ax1.legend(ncol=2, fontsize=8)
light_grid(ax1); add_panel_label(ax1, "(B)", y=1.08)

ax2=fig.add_subplot(gs[2])
ax2.plot(ts, -psi_all, color=C["psi"], lw=1.3)
ax2.fill_between(ts, -psi_all, alpha=0.10, color=C["psi"])
ax2.set_ylabel("|ψ| Matric Suction (m)")
ax2.set_title("C — Matric Suction (Van Genuchten)", fontweight="bold", loc="left")
p2c, p98c = np.percentile(-psi_all, [1, 99])
ax2.set_ylim(max(0, p2c-0.1), p98c+0.5)
light_grid(ax2); add_panel_label(ax2, "(C)", y=1.08)

ax3=fig.add_subplot(gs[3])
ax3.plot(ts, u_all, color=C["u"], lw=1.3)
ax3.axhline(0, color="gray", lw=0.7, ls=":")
ax3.fill_between(ts, u_all, 0, where=(u_all>0),
                 color=C["warn"], alpha=0.3, label="Positive u (destabilising)")
ax3.fill_between(ts, u_all, 0, where=(u_all<=0),
                 color=C["fos"], alpha=0.15, label="Negative u (stabilising)")
ax3.set_ylabel("u  Pore Pressure (kPa)")
ax3.set_title("D — Pore Water Pressure", fontweight="bold", loc="left")
ax3.legend(fontsize=8)
p2d, p98d = np.percentile(u_all, [1, 99])
ax3.set_ylim(p2d - 2, p98d + 2)
light_grid(ax3); add_panel_label(ax3, "(D)", y=1.08)

ax4=fig.add_subplot(gs[4])
ax4.plot(ts, FoS_all, color=C["fos"], lw=1.4, label="FoS")
ax4.axhline(1.0, color=C["warn"],    lw=1.8, ls="--", label="FoS = 1.0  ⚠ Failure")
ax4.axhline(1.3, color="darkorange", lw=1.2, ls=":",  label="FoS = 1.3  Warning")
ax4.fill_between(ts, FoS_all, 1.0, where=(FoS_all<1.0), color=C["warn"], alpha=0.4)
ax4.set_ylabel("Factor of Safety (−)")
ax4.set_xlabel("Timestamp")
ax4.set_title("E — Factor of Safety (Infinite Slope Model)", fontweight="bold", loc="left")
ax4.legend(ncol=3, fontsize=8)
p2e, p98e = np.percentile(FoS_all, [1, 99])
ax4.set_ylim(max(0, p2e-0.2), min(10, p98e+0.5))
light_grid(ax4); add_panel_label(ax4, "(E)", y=1.08)

for ax in [ax0,ax1,ax2,ax3,ax4]:
    ax.set_xlim(ts[0], ts[-1])
    for t in ax.get_xticklabels(): t.set_rotation(15)

fig.suptitle(
    "Physics-Informed Machine Learning (PIML) — Slope Stability Monitoring\n"
    f"Dev108 (Sandy Clay Loam, x=7 m)  |  "
    f"Slope={SLOPE}°   c'={c_} kPa   φ'=28°   H={H} m",
    fontsize=12, fontweight="bold", y=1.002)
savefig(fig, "fig10_full_dashboard.png")

# ═══════════════════════════════════════════════════════════
# RESULTS SUMMARY
# ═══════════════════════════════════════════════════════════
print("\n" + "="*60)
print("   PIML SLOPE STABILITY — RESULTS SUMMARY")
print("="*60)
print(f"  Records analysed     : {len(df):,}")
print(f"  Train R²             : {r2_tr:.4f}")
print(f"  Val   R²             : {r2_val:.4f}")
print(f"  Test  R²             : {r2_te:.4f}")
print(f"  Test  RMSE (θ)       : {rmse_te:.5f} m³/m³")
print(f"  VG θ_r / θ_s         : {VG['theta_r']} / {VG['theta_s']}")
print(f"  VG α / n             : {VG['alpha']} m⁻¹ / {VG['n']}")
print(f"  ψ  range             : {psi_all.min():.2f} to {psi_all.max():.2f} m")
print(f"  u  range             : {u_all.min():.2f} to {u_all.max():.2f} kPa")
print(f"  FoS mean             : {FoS_all.mean():.3f}")
print(f"  FoS min              : {FoS_all.min():.3f}")
print(f"  ⚠ FoS < 1.0 events  : {(FoS_all<1.0).sum()}")
print(f"  ⚠ FoS < 1.3 events  : {(FoS_all<1.3).sum()}")
print(f"  Learned αD           : {alpha_D:.4f}")
print(f"  Physics residual σ   : {sd_r:.4e}")
print("="*60)
print(f"\n✅ All figures saved to: {OUT}/")
print(f"✅ Model saved to      : {model_path}")

Device: cpu

[Step 1-2] Loading data ...
  Artifact rows removed: 246 (theta ≤ 0.30)
  Rows after subsample : 30,854

[Step 3] Building PIML model ...
  Parameters: 6,466

[Step 4-6] Training ...
  Epoch  100 | Train 0.00031 | Val 0.00007 | αD=0.2742
  Epoch  200 | Train 0.00018 | Val 0.00004 | αD=0.2228
  Epoch  300 | Train 0.00013 | Val 0.00004 | αD=0.1898
  Epoch  400 | Train 0.00007 | Val 0.00003 | αD=0.1671
  Epoch  500 | Train 0.00005 | Val 0.00003 | αD=0.1507
  Epoch  600 | Train 0.00003 | Val 0.00002 | αD=0.1385
  Epoch  700 | Train 0.00001 | Val 0.00002 | αD=0.1292
  Epoch  800 | Train 0.00001 | Val 0.00002 | αD=0.1221
  Epoch  900 | Train 0.00001 | Val 0.00002 | αD=0.1165
  Epoch 1000 | Train 0.00001 | Val 0.00002 | αD=0.1122
  Epoch 1100 | Train 0.00001 | Val 0.00002 | αD=0.1088
  Epoch 1200 | Train 0.00001 | Val 0.00002 | αD=0.1062
  Epoch 1300 | Train 0.00001 | Val 0.00002 | αD=0.1042
  Epoch 1400 | Train 0.00001 | Val 0.00002 | αD=0.1027
  Epoch 1500 | Train 0.00001 | Val

In [ ]:
# training loop এর ভেতরে, প্রথম batch এ
print("dthdt_pred mean:", dthdt_pred.abs().mean().item())
print("I_n mean:", I_n.abs().mean().item())
print("D_n mean:", D_n.abs().mean().item())

dthdt_pred mean: 1.5744154424623957e-08
I_n mean: 0.0
D_n mean: 9.613681868358981e-06


In [ ]:
print("pred max:", pred.max().item())
print("pred min:", pred.min().item())
print("pred_real max:", pred_real.max().item())
print("pred_real min:", pred_real.min().item())
print("(pred-pred_prev) mean:", (pred-pred_prev).abs().mean().item())

pred max: 0.5063881278038025
pred min: 0.38469794392585754
pred_real max: 0.4250226318836212
pred_real min: 0.42402130365371704


TypeError: unsupported operand type(s) for -: 'numpy.ndarray' and 'Tensor'